In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, KFold, cross_validate, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
from google.colab import files
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

In [20]:
RANDOM_STATE = 123
generated_files = []

In [21]:
# =====================================================================
# HELPER FUNCTIONS FOR PREDICTION TASK
# =====================================================================
def save_result(data, path):
    """Save a DataFrame to CSV and track it for Colab downloading."""
    try:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(path, index=False)
        print(f"Saved to {path}")
        generated_files.append(path)
    except Exception as e:
        print(f"Error saving data: {e}")

def preprocess_prediction_features(X_train, X_test):
    """Explicitly impute missing values and scale/encode features (No Leakage)."""
    num_cols = X_train.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    # Imputation
    num_imputer = SimpleImputer(strategy="median")
    cat_imputer = SimpleImputer(strategy="most_frequent")

    X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
    X_test_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

    # Categorical Encoding
    if cat_cols:
        X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[cat_cols]), columns=cat_cols, index=X_train.index)
        X_test_cat = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]), columns=cat_cols, index=X_test.index)

        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train_cat), columns=encoder.get_feature_names_out(cat_cols), index=X_train.index)
        X_test_encoded = pd.DataFrame(encoder.transform(X_test_cat), columns=encoder.get_feature_names_out(cat_cols), index=X_test.index)
    else:
        X_train_encoded = pd.DataFrame(index=X_train.index)
        X_test_encoded = pd.DataFrame(index=X_test.index)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_num), columns=num_cols, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_num), columns=num_cols, index=X_test.index)

    # Recombine
    X_train_processed = pd.concat([X_train_scaled, X_train_encoded], axis=1)
    X_test_processed = pd.concat([X_test_scaled, X_test_encoded], axis=1)

    return X_train_processed, X_test_processed

In [22]:
def evaluate_predictions(y_true, y_pred):
    """Compute prediction metrics."""
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
        "n_test": int(len(y_true)),
    }

def build_feature_importance_df(model, feature_names):
    """Build a feature-importance table for tree-based models."""
    if not hasattr(model, "feature_importances_"):
        return None
    return pd.DataFrame({"feature": feature_names, "importance": model.feature_importances_}).sort_values("importance", ascending=False)

In [28]:
# =====================================================================
# MAIN PIPELINE
# =====================================================================
def run_hybrid_pipeline():
    class_id = "1tW6Si8XiQJclYrP_1be6Td7tU7DbLyGG"
    pred_id = "13vSsB9ys6R4eumq6Z51Ha7MpEDHt7M6u"

    class_url = f"https://drive.google.com/uc?export=download&id={class_id}"
    pred_url = f"https://drive.google.com/uc?export=download&id={pred_id}"

    # =====================================================================
    # TASK 1: CLASSIFICATION (Predicting Deep vs. Shallow)
    # =====================================================================
    print(f"{'='*60}")
    print("--- 5-Fold Cross-Validation: CLASSIFICATION TASK ---")
    print(f"{'='*60}")

    df_class = pd.read_csv(class_url)

    # Target Encoding
    if df_class['Foundation Type'].dtype == 'object':
        df_class['Foundation Type'] = df_class['Foundation Type'].map({'Deep': 1, 'Shallow': 0})
    df_class.dropna(subset=['Foundation Type'], inplace=True)

    X_class = df_class.drop(columns=['Foundation Type'])
    y_class = df_class['Foundation Type']

    # Preprocessor Setup
    num_cols_c = X_class.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
    cat_cols_c = X_class.select_dtypes(include=['object', 'category']).columns.tolist()

    preprocessor_c = ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols_c),
            ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_c)
        ])

    # Classification Models
    models_class = {
        "Logistic Regression": LogisticRegression(random_state=42),
        "Support Vector Machine": SVC(kernel='linear', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
        "LightGBM (Leashed)": lgb.LGBMClassifier(max_depth=3, num_leaves=7, min_child_samples=5, n_estimators=50, random_state=42, verbose=-1)
    }

    cv_class = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring_class = ['accuracy', 'precision', 'recall', 'f1']

    for name, model in models_class.items():
        clf = Pipeline(steps=[('preprocessor', preprocessor_c), ('classifier', model)])
        cv_results = cross_validate(clf, X_class, y_class, cv=cv_class, scoring=scoring_class)
        print(f"\nModel: {name}")
        print(f"  -> Accuracy:  {np.mean(cv_results['test_accuracy']):.3f}  (+/- {np.std(cv_results['test_accuracy']):.3f})")
        print(f"  -> Precision: {np.mean(cv_results['test_precision']):.3f}")
        print(f"  -> Recall:    {np.mean(cv_results['test_recall']):.3f}")
        print(f"  -> F1 Score:  {np.mean(cv_results['test_f1']):.3f}")

    # ---------------------------------------------------------------------
    # PART 2: PREDICTION (Grid Search Method)
    # ---------------------------------------------------------------------
    print(f"\n\n{'='*60}")
    print("--- Professor's Method: PREDICTION TASK ---")
    print(f"{'='*60}")

    df_pred = pd.read_csv(pred_url)
    df_pred.dropna(subset=['Pier total Depth (ft)'], inplace=True)

    X_pred = df_pred.drop(columns=['Pier total Depth (ft)'])
    y_pred = df_pred['Pier total Depth (ft)']

    # 80/20 Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_pred, y_pred, test_size=0.2, random_state=RANDOM_STATE)

    # Explicit Preprocessing
    X_train_processed, X_test_processed = preprocess_prediction_features(X_train, X_test)

    # Models to test with GridSearch
    model_configs = {
        "ridge": (Ridge(), {"alpha": [0.1, 1.0, 10.0]}),
        "svr": (SVR(), {"C": [0.1, 1.0, 10.0], "epsilon": [0.1, 0.5, 1.0], "kernel": ["rbf"]}),
        "random_forest": (RandomForestRegressor(random_state=RANDOM_STATE), {"n_estimators": [50, 100], "max_depth": [3, 5]}),
        "lightgbm": (lgb.LGBMRegressor(random_state=RANDOM_STATE, verbose=-1, min_child_samples=5), {"max_depth": [3, 5], "num_leaves": [7, 15]})
    }

    cv_pred = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    metrics_rows = []
    prediction_tables = {}
    fitted_models = {}

    for name, (estimator, param_grid) in model_configs.items():
        search = GridSearchCV(estimator=estimator, param_grid=param_grid, scoring="neg_root_mean_squared_error", cv=cv_pred, n_jobs=-1)
        search.fit(X_train_processed, y_train)

        best_model = search.best_estimator_
        y_pred_vals = best_model.predict(X_test_processed)

        metrics = evaluate_predictions(y_test, y_pred_vals)
        metrics["model"] = name
        metrics["best_params"] = str(search.best_params_)
        metrics_rows.append(metrics)

        pred_df = X_test.copy()
        pred_df["observed_Pier total Depth (ft)"] = y_test.values
        pred_df["predicted_Pier total Depth (ft)"] = y_pred_vals
        pred_df["model"] = name

        prediction_tables[name] = pred_df
        fitted_models[name] = best_model

    # Process and Output Results
    metrics_df = pd.DataFrame(metrics_rows).sort_values("rmse", ascending=True).reset_index(drop=True)
    predictions_df = pd.concat(prediction_tables.values(), ignore_index=True)

    best_model_name = metrics_df.loc[0, "model"]
    best_predictions_df = prediction_tables[best_model_name].reset_index(drop=True)

    base_out = "output_accuracy_prediction"
    save_result(metrics_df, f"{base_out}_model_comparison_metrics.csv")
    save_result(predictions_df, f"{base_out}_all_model_predictions.csv")
    save_result(best_predictions_df, f"{base_out}_best_model_predictions.csv")

    for model_name in ["random_forest", "lightgbm"]:
        if model_name in fitted_models:
            imp_df = build_feature_importance_df(fitted_models[model_name], X_train_processed.columns.tolist())
            if imp_df is not None:
                save_result(imp_df, f"{base_out}_feature_importance_{model_name}.csv")

    print(f"\nBest Prediction Model: {best_model_name}")
    print(metrics_df.to_string(index=False))

    print("\nInitiating downloads for all Prediction task files...")
    for file_path in generated_files:
        files.download(file_path)

In [29]:
if __name__ == "__main__":
    run_hybrid_pipeline()

--- 5-Fold Cross-Validation: CLASSIFICATION TASK ---

Model: Logistic Regression
  -> Accuracy:  0.971  (+/- 0.035)
  -> Precision: 0.967
  -> Recall:    1.000
  -> F1 Score:  0.983

Model: Support Vector Machine
  -> Accuracy:  0.957  (+/- 0.035)
  -> Precision: 0.967
  -> Recall:    0.983
  -> F1 Score:  0.974

Model: Random Forest
  -> Accuracy:  0.971  (+/- 0.035)
  -> Precision: 0.967
  -> Recall:    1.000
  -> F1 Score:  0.983

Model: LightGBM (Leashed)
  -> Accuracy:  0.986  (+/- 0.029)
  -> Precision: 0.983
  -> Recall:    1.000
  -> F1 Score:  0.991


--- Professor's Method: PREDICTION TASK ---
Saved to output_accuracy_prediction_model_comparison_metrics.csv
Saved to output_accuracy_prediction_all_model_predictions.csv
Saved to output_accuracy_prediction_best_model_predictions.csv
Saved to output_accuracy_prediction_feature_importance_random_forest.csv
Saved to output_accuracy_prediction_feature_importance_lightgbm.csv

Best Prediction Model: random_forest
     mae      rmse  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>